# 05 — Run SOLWEIG and create the thermal field

This notebook runs SOLWEIG over the PoC domain.

The main run uses a PVGIS EPW **typical meteorological year (TMY)** for July 15. This is suitable for testing the method, but it is not an observed Sevilla heat-event simulation.

The current downstream PoC uses `utci_max.tif`, the cell-wise daily maximum UTCI surface. Later notebooks show why a common hot-afternoon time slice would be preferable for a publishable city-scale analysis.

In [ ]:
from pathlib import Path

# The notebooks_v02 folder is expected to sit directly inside the repository.
# If you open Jupyter from another working directory, set PROJECT_ROOT manually.
HERE = Path.cwd().resolve()
PROJECT_ROOT = HERE.parent if HERE.name == "notebooks_v02" else HERE

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
OUTPUTS = PROJECT_ROOT / "outputs"

for folder in [DATA_RAW, DATA_INTERIM, OUTPUTS]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

In [ ]:
import pandas as pd
import solweig

SOLWEIG_INPUTS = DATA_INTERIM / "solweig"
SOLWEIG_CACHE = DATA_INTERIM / "solweig_cache"
SOLWEIG_OUTPUTS = OUTPUTS / "solweig"
WEATHER_DIR = DATA_INTERIM / "weather"

for folder in [SOLWEIG_CACHE, SOLWEIG_OUTPUTS, WEATHER_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

## 1. Prepare the SOLWEIG surface

In [ ]:
surface = solweig.SurfaceData.prepare(
    dsm=SOLWEIG_INPUTS / "dsm.tif",
    cdsm=SOLWEIG_INPUTS / "cdsm.tif",
    dem=SOLWEIG_INPUTS / "dem.tif",
    working_dir=SOLWEIG_CACHE,
)

## 2. Download and read the Sevilla EPW weather file

In [ ]:
epw_path = solweig.download_epw(
    latitude=37.39,
    longitude=-5.96,
    output_path=WEATHER_DIR / "sevilla.epw",
)

weather_list = solweig.Weather.from_epw(
    epw_path,
    start="07-15",
    end="07-15",
)

location = solweig.Location.from_epw(epw_path)

print("Weather timesteps:", len(weather_list))
print("First timestamp:", weather_list[0].datetime)

## 3. Validate and run SOLWEIG

In [ ]:
warnings = solweig.validate_inputs(
    surface,
    location,
    weather_list,
)

for warning in warnings:
    print(warning)

summary = solweig.calculate(
    surface=surface,
    weather=weather_list,
    location=location,
    output_dir=SOLWEIG_OUTPUTS,
    outputs=["tmrt", "utci", "shadow"],
)

print(summary.report())

## 4. Inspect the hourly thermal series

In [ ]:
ts = summary.timeseries

hourly = pd.DataFrame({
    "datetime": ts.datetime,
    "ta": ts.ta,
    "tmrt_mean": ts.tmrt_mean,
    "utci_mean": ts.utci_mean,
    "sun_fraction": ts.sun_fraction,
})

display(hourly)

print("\nPeak mean UTCI timestep:")
display(hourly.loc[[hourly["utci_mean"].idxmax()]])

## Interpretation note

The thermal simulation is deliberately a PoC stress-test. The EPW represents a typical meteorological year, and the daily-maximum surface can combine different peak times across cells. Keep these caveats attached to any later accessibility result.